<a href="https://colab.research.google.com/github/Saransh-Basu-01/50-Ml-Projects-to-understand-LLMs/blob/main/Project_22_Evaluating_models_with_hellaswag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from datasets import load_dataset

from statsmodels.stats.contingency_tables import mcnemar

# pytorch-related libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

# libraries to import models
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import RobertaTokenizer, RobertaForMaskedLM

# Part_1:Import and Compare GPT-2 and Roberta Model

## Task_1:
- Import four models: GPT-2-small, GPT-2-large, RoBERTa-base,
and RoBERTa-large. The two GPT-2 variants use the same tok
enizer, and the two RoBERTa variants use the same tokenizer. Make
sure all models and tokenizers have interpretable variable names,
e.g., gpt2_s and tokenizerG.
Switch the models to evaluation mode and move them to the
GPU.
Write code to create the table below (Xfmr is “transformer”;
the number of transformer layers)

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
tokenizerG = AutoTokenizer.from_pretrained('gpt2')
gpt2_s = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
gpt2_l = AutoModelForCausalLM.from_pretrained('gpt2-large').to(device)
gpt2_s.eval()
gpt2_l.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [4]:
tokenizerR = RobertaTokenizer.from_pretrained('roberta-base')
roberta_s = RobertaForMaskedLM.from_pretrained('roberta-base').to(device)
roberta_l = RobertaForMaskedLM.from_pretrained('roberta-large').to(device)

roberta_s.eval()
roberta_l.eval()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (

In [5]:
roberta_s.config

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.21.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

In [6]:
print(' Model | Tokens | Emb.dim | Xfmr | Parameters')
print('-------+--------+---------+------+-------------')

# GPT2-small
print(f'GPT2-S | {tokenizerG.vocab_size:,} |   {gpt2_s.config.n_embd:4}  |  {gpt2_s.config.n_layer}  | {gpt2_s.num_parameters():,}')

# GPT2-large
print(f'GPT2-L | {tokenizerG.vocab_size:,} |   {gpt2_l.config.n_embd:4}  |  {gpt2_l.config.n_layer}  | {gpt2_l.num_parameters():,}')

# roberta-small
print(f'RoBA-S | {tokenizerR.vocab_size:,} |   {roberta_s.config.hidden_size:4}  |  {roberta_s.config.num_hidden_layers}  | {roberta_s.num_parameters():,}')

# roberta-large
print(f'RoBA-L | {tokenizerR.vocab_size:,} |   {roberta_l.config.hidden_size:4}  |  {roberta_l.config.num_hidden_layers}  | {roberta_l.num_parameters():,}')


 Model | Tokens | Emb.dim | Xfmr | Parameters
-------+--------+---------+------+-------------
GPT2-S | 50,257 |    768  |  12  | 124,439,808
GPT2-L | 50,257 |   1280  |  36  | 774,030,080
RoBA-S | 50,265 |    768  |  12  | 124,697,433
RoBA-L | 50,265 |   1024  |  24  | 355,412,057


# Part_2:Import and Export HellaSwag

## Task_1:
- Import the HellaSwag validation set using the load_dataset function
in the datasets library. Examine one sample. Guess the meaning of
the keys ctx_a, ctx_b, ctx, endings, and label.

In [7]:
dataset = load_dataset('allenai/hellaswag',split='validation')
dataset


README.md:   0%|          | 0.00/823 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.09MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.29MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/39905 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

Dataset({
    features: ['ind', 'activity_label', 'ctx_a', 'ctx_b', 'ctx', 'endings', 'source_id', 'split', 'split_type', 'label'],
    num_rows: 10042
})

In [8]:
dataset[1]

{'ind': 92,
 'activity_label': 'Clean and jerk',
 'ctx_a': 'A lady walks to a barbell. She bends down and grabs the pole.',
 'ctx_b': 'the lady',
 'ctx': 'A lady walks to a barbell. She bends down and grabs the pole. the lady',
 'endings': ['swings and lands in her arms.',
  'pulls the barbell forward.',
  'pulls a rope attached to the barbell.',
  'stands and lifts the weight over her head.'],
 'source_id': 'activitynet~v_-lJS58hyo1c',
 'split': 'val',
 'split_type': 'zeroshot',
 'label': '3'}

# Part_3:Visualize the evaluation process

## Task_1:
- Create two prompts by concatenating the context with two of the
four endings: one with the target ending and one with an incorrect
ending. Tokenize the prompts using the GPT-2 tokenizer

In [14]:
int(dataset[1]['label'])

3

In [17]:
dataset[224]

{'ind': 2795,
 'activity_label': 'Drum corps',
 'ctx_a': 'People watch a parade where people walk on front a marching band holding a banner and waving the viewers.',
 'ctx_b': 'a man',
 'ctx': 'People watch a parade where people walk on front a marching band holding a banner and waving the viewers. a man',
 'endings': ['cross the street holding a small flag and extend his hand.',
  'is in the middle of the parade playing a bicycle.',
  'walks out while wearing a tutu in the parade while others walk on front.',
  'wearing an orange jacket falls over from the fall, they help him up.'],
 'source_id': 'activitynet~v_XKA2W2735WM',
 'split': 'val',
 'split_type': 'indomain',
 'label': '0'}

In [16]:
# pick a random example
exampleNum = 224
target = int(dataset[exampleNum]['label'])

# context tokens and length
context = dataset[exampleNum]['ctx']
context_len =len(context)

# prompts and their lengths
promptC = f"{context} {dataset[exampleNum]['endings'][target]}"
promptC_tox =tokenizerG.encode(promptC,return_tensors="pt")
promptC_len =len(promptC_tox[0])

promptI = f"{context} {dataset[exampleNum]['endings'][3-target]}"
promptI_tox = tokenizerG.encode(promptI,return_tensors='pt')
promptI_len = len( promptI_tox[0] )

# show the prompts
print(f'Context:\n   "{context}"\n')
print(f'Correct ending:\n   "{promptC}"\n')
print(f'Incorrect ending:\n   "{promptI}"')

Context:
   "People watch a parade where people walk on front a marching band holding a banner and waving the viewers. a man"

Correct ending:
   "People watch a parade where people walk on front a marching band holding a banner and waving the viewers. a man cross the street holding a small flag and extend his hand."

Incorrect ending:
   "People watch a parade where people walk on front a marching band holding a banner and waving the viewers. a man wearing an orange jacket falls over from the fall, they help him up."


## Task_2:
- Run both prompts through the GPT-2-small model. Log-softmax
transform the logits.
Create an array of log-softmax prediction values for each to
ken. This is created using a for-loop over the tokens, in which the
log-softmax corresponding to the next-token index is extracted. The
result of this step will be two NumPy arrays comprising N −1 tokens
for each prompt, where the value in each element is the log-softmax
probability of the subsequent token.
Calculate the sentence-level probability as the average of the
log-probabilities for the ending tokens (exclude the context tokens).
Print the scores and see whether the target ending has a higher
probability than the non-target ending.
Answer the following questions: (1) What is the logic of sum
ming the log-probability scores and then dividing by the number of
tokens? (2) Is it necessary to exclude the context token probabilities?